# 🧠 EEG Data Exploration

This notebook explores the **PhysioNet EEG Motor Movement/Imagery Dataset** (Schalk et al., 2004).

**Dataset Details:**
- 109 subjects, 64 EEG channels, 160 Hz sampling rate
- 4 motor imagery tasks: left fist, right fist, both fists, both feet

We will cover:
1. Downloading & loading data with MNE
2. Visualizing raw EEG trials
3. Class-averaged signals (Event-Related Desynchronization)
4. Class distribution analysis

In [1]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.eeg.dataset import download_physionet_data, load_subject_data, CLASS_NAMES
from src.visualization.eeg_plots import plot_eeg_trial, plot_class_averaged_signals

## 1. Downloading and Loading Data

We use MNE-Python to download EEG data from PhysioNet. The data is cached locally after the first download.

In [2]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Download data for Subject 1 (will be cached after first run)
download_physionet_data(DATA_DIR, subjects=[1])

# Load preprocessed epochs — returns (X, y)
# X: (n_epochs, n_channels, n_times)
# y: (n_epochs,) with labels 0=left_fist, 1=right_fist, 2=both_fists, 3=both_feet
X, y = load_subject_data(
    subject_id=1,
    data_dir=DATA_DIR,
    tmin=0.0,
    tmax=4.0,
    bandpass=(4.0, 40.0),
    resample_freq=128.0,
)

print(f"Data shape: {X.shape}  (epochs, channels, time_points)")
print(f"Labels shape: {y.shape}")
print(f"\nSampling rate after resampling: 128 Hz")
print(f"Epoch duration: {X.shape[-1] / 128.0:.2f} seconds")
print(f"\nClass distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (y == i).sum()
    print(f"  {name}: {count} trials ({100*count/len(y):.1f}%)")

Data shape: (90, 64, 513)  (epochs, channels, time_points)
Labels shape: (90,)

Sampling rate after resampling: 128 Hz
Epoch duration: 4.01 seconds

Class distribution:
  left_fist: 23 trials (25.6%)
  right_fist: 22 trials (24.4%)
  both_fists: 21 trials (23.3%)
  both_feet: 24 trials (26.7%)


## 2. Visualizing Raw EEG Trials

EEG signals are noisy, low-amplitude, and multi-channel. Let's look at a few key channels over the motor cortex (C3, Cz, C4) for one trial.

In [3]:
# Show the first few channels of the first trial
n_channels_to_show = min(8, X.shape[1])
sfreq = 128.0

print(f"Trial 0 — Class: {CLASS_NAMES[y[0]]}")
plot_eeg_trial(
    X[0],
    sfreq=sfreq,
    n_channels_to_show=n_channels_to_show,
    title=f"EEG Trial 0 — {CLASS_NAMES[y[0]]}",
)

Trial 0 — Class: right_fist


## 3. Class-Averaged Signals

Averaging across trials of the same class reveals the **Event-Related Desynchronization** (ERD) patterns that distinguish motor imagery tasks. For left vs right imagery, differences should be visible between left (C3) and right (C4) motor cortex channels.

In [4]:
# Average signals across the first 4 channels
channels_to_plot = list(range(min(4, X.shape[1])))

plot_class_averaged_signals(
    X, y, CLASS_NAMES,
    channels=channels_to_plot,
    sfreq=sfreq,
)

## 4. Next Steps

Now that we've explored the raw data, we can:
1. Feed it through our preprocessing pipeline (`src/eeg/preprocessing.py`)
2. Apply augmentation (`src/eeg/augmentation.py`)
3. Tokenize with the CNN front-end (`src/eeg/tokenizer.py`)
4. Train the EEG-Transformer! (`scripts/train_eeg.py`)